# Workshop-3 — Paso 2: Limpieza y armonización de datos

**Qué pide el taller en este paso:** estandarizar nombres de columnas, estandarizar tipos de datos, tratar valores faltantes, resolver inconsistencias de esquema y unir los datasets en uno solo. Además, **justificar cada decisión de limpieza**.

**Entrada:** los 5 CSV de `data/raw/` (no se modifican).
**Salida:** `data/processed/happiness_unified.csv`

Cada sección aplica una de las decisiones que salieron del EDA (`eda.ipynb`, secciones 8 y 9). Al final hay una tabla resumen con todas las decisiones y su justificación.

## 1. Cargar los archivos crudos

In [1]:
import pandas as pd

RAW = "../data/raw"
PROCESSED = "../data/processed"
YEARS = [2015, 2016, 2017, 2018, 2019]

dfs = {year: pd.read_csv(f"{RAW}/{year}.csv") for year in YEARS}
print("Filas por año:", {year: len(df) for year, df in dfs.items()})
print("Total de filas:", sum(len(df) for df in dfs.values()))

Filas por año: {2015: 158, 2016: 157, 2017: 155, 2018: 156, 2019: 156}
Total de filas: 782


## 2. Estandarizar nombres de columnas y resolver las diferencias de esquema

**Decisión:** escribir a mano un diccionario de equivalencias por cada "familia" de esquema (2015-2016, 2017, 2018-2019) y quedarse solo con las columnas que existen en los 5 años.

**Justificación:**
- Un diccionario explícito deja a la vista qué columna original alimenta cada columna final. Es fácil de revisar y de auditar.
- Los nombres finales son los mismos del JSON que exige el taller para Kafka, así el productor no tiene que renombrar nada.
- Las columnas que no están en los 5 años (región, error estándar, intervalos de confianza, whiskers, residuo de distopía) se descartan porque dejarían años enteros vacíos.
- El ranking también se descarta: se calcula a partir del puntaje, y usarlo para predecir el puntaje sería fuga del objetivo.

In [2]:
# columna original -> columna unificada
MAPA_2015_2016 = {
    "Country": "country",
    "Economy (GDP per Capita)": "gdp",
    "Family": "family",
    "Health (Life Expectancy)": "health",
    "Freedom": "freedom",
    "Generosity": "generosity",
    "Trust (Government Corruption)": "corruption",
    "Happiness Score": "happiness_score",
}

MAPA_2017 = {
    "Country": "country",
    "Economy..GDP.per.Capita.": "gdp",
    "Family": "family",
    "Health..Life.Expectancy.": "health",
    "Freedom": "freedom",
    "Generosity": "generosity",
    "Trust..Government.Corruption.": "corruption",
    "Happiness.Score": "happiness_score",
}

MAPA_2018_2019 = {
    "Country or region": "country",
    "GDP per capita": "gdp",
    "Social support": "family",
    "Healthy life expectancy": "health",
    "Freedom to make life choices": "freedom",
    "Generosity": "generosity",
    "Perceptions of corruption": "corruption",
    "Score": "happiness_score",
}

MAPAS = {2015: MAPA_2015_2016, 2016: MAPA_2015_2016, 2017: MAPA_2017,
         2018: MAPA_2018_2019, 2019: MAPA_2018_2019}

# Orden final de las columnas
COLUMNAS = ["country", "year", "gdp", "family", "health",
            "freedom", "generosity", "corruption", "happiness_score"]

## 3. Crear la columna `year` y unir los 5 años

**Decisión:** crear `year` a partir del nombre del archivo y apilar los 5 años uno debajo del otro.

**Justificación:** ningún archivo trae el año adentro. Sin esa columna, al unirlos no se sabría a qué año pertenece cada fila.

In [3]:
partes = []
for year, df in dfs.items():
    mapa = MAPAS[year]
    parte = df[list(mapa)].rename(columns=mapa)   # solo las columnas comunes, ya renombradas
    parte["year"] = year                          # el año sale del nombre del archivo
    partes.append(parte[COLUMNAS])

unificado = pd.concat(partes, ignore_index=True)
print(unificado.shape)
unificado.head()

(782, 9)


,country,year,gdp,family,health,freedom,generosity,corruption,happiness_score
0,Switzerland,2015,1.39651,1.34951,0.94143,0.66557,0.29678,0.41978,7.587
1,Iceland,2015,1.30232,1.40223,0.94784,0.62877,0.43630,0.14145,7.561
2,Denmark,2015,1.32548,1.36058,0.87464,0.64938,0.34139,0.48357,7.527
3,Norway,2015,1.45900,1.33095,0.88521,0.66973,0.34699,0.36503,7.522
4,Canada,2015,1.32629,1.32261,0.90563,0.63297,0.45811,0.32957,7.427


## 4. Unificar los nombres de países

**Decisión:** quitar espacios sobrantes y dejar un solo nombre para los 6 países que aparecen escritos de dos formas.

**Justificación:** el país será la clave del evento en Kafka y una fila de `dim_country`. Si un mismo país tiene dos nombres, quedaría como dos países distintos en la dimensión y sus eventos podrían caer en particiones diferentes.

No se tocan los países que simplemente no fueron medidos todos los años: eso es cobertura de la encuesta, no un error.

In [4]:
PAISES = {
    "Hong Kong S.A.R., China": "Hong Kong",
    "Taiwan Province of China": "Taiwan",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "North Cyprus": "Northern Cyprus",
    "Macedonia": "North Macedonia",
    "Somaliland region": "Somaliland Region",
}

antes = unificado["country"].nunique()
unificado["country"] = unificado["country"].str.strip().replace(PAISES)
despues = unificado["country"].nunique()

print("Países distintos antes:", antes)
print("Países distintos después:", despues)

Países distintos antes: 170
Países distintos después: 164


## 5. Tratar el valor faltante

**Decisión:** rellenar la única celda vacía (corrupción de United Arab Emirates en 2018) con la **mediana de corrupción de 2018**.

**Justificación:**
- Es una sola celda entre 782 filas. Eliminar la fila borraría un país completo por un solo dato.
- Se usa la **mediana y no el promedio** porque el EDA mostró 67 valores atípicos en corrupción; el promedio se deja arrastrar por esos valores altos y la mediana no.
- Se usa la mediana **del mismo año (2018)** para no mezclar mediciones de años distintos.

In [5]:
print("Celdas vacías antes:")
print(unificado.isna().sum()[unificado.isna().sum() > 0])

mediana_2018 = unificado.loc[unificado["year"] == 2018, "corruption"].median()
promedio_2018 = unificado.loc[unificado["year"] == 2018, "corruption"].mean()
print()
print("Mediana de corrupción 2018:", round(mediana_2018, 5))
print("Promedio de corrupción 2018:", round(promedio_2018, 5), "(más alto, arrastrado por los atípicos)")

unificado["corruption"] = unificado["corruption"].fillna(mediana_2018)

print()
print("Celdas vacías después:", unificado.isna().sum().sum())
unificado[(unificado["country"] == "United Arab Emirates") & (unificado["year"] == 2018)]

Celdas vacías antes:
corruption    1
dtype: int64

Mediana de corrupción 2018: 0.082
Promedio de corrupción 2018: 0.112 (más alto, arrastrado por los atípicos)

Celdas vacías después: 0


,country,year,gdp,family,health,freedom,generosity,corruption,happiness_score
489,United Arab Emirates,2018,2.096,0.776,0.67,0.284,0.186,0.082,6.774


## 6. Estandarizar tipos de datos y precisión

**Decisión:** `country` como texto, `year` como entero, los 7 indicadores como decimales redondeados a **5 decimales**.

**Justificación:**
- Los tipos coinciden con el contrato del evento que define el taller (país texto, año entero, indicadores numéricos).
- 5 decimales es la mayor precisión real de las fuentes (2015 y 2016). Redondear ahí elimina el "ruido" de 2017 (valores como `7.53700017929077`) sin perder información de ningún año.

In [6]:
NUMERICAS = ["gdp", "family", "health", "freedom", "generosity", "corruption", "happiness_score"]

unificado["country"] = unificado["country"].astype("string")
unificado["year"] = unificado["year"].astype("int64")
unificado[NUMERICAS] = unificado[NUMERICAS].astype("float64").round(5)

unificado.dtypes

country             string
year                 int64
gdp                float64
family             float64
health             float64
freedom            float64
generosity         float64
corruption         float64
happiness_score    float64
dtype: object

## 7. Decisiones de "no hacer nada" (también se justifican)

- **Valores atípicos:** se conservan. Son valores reales de países reales, no errores de digitación. Quitarlos haría que el modelo no conociera los casos extremos que sí llegarán por Kafka.
- **Ceros exactos (31):** se conservan. Son el valor mínimo del indicador en ese año, no datos faltantes.
- **Duplicados:** no había. Se vuelve a comprobar abajo, porque al unificar nombres de países podría aparecer un país repetido en un mismo año.

## 8. Validación final

In [7]:
filas_repetidas = unificado.duplicated().sum()
pais_anio_repetido = unificado.duplicated(subset=["country", "year"]).sum()

print("Tamaño:", unificado.shape)
print("Celdas vacías:", unificado.isna().sum().sum())
print("Filas repetidas:", filas_repetidas)
print("Combinaciones país-año repetidas:", pais_anio_repetido)
print("Valores negativos:", (unificado[NUMERICAS] < 0).sum().sum())
print()
print("Filas por año:")
print(unificado["year"].value_counts().sort_index())

# Si alguna de estas condiciones no se cumple, el notebook se detiene con error
assert unificado.shape == (782, 9), "El tamaño no es el esperado"
assert unificado.isna().sum().sum() == 0, "Quedaron celdas vacías"
assert pais_anio_repetido == 0, "Hay un país repetido en un mismo año"
assert list(unificado.columns) == COLUMNAS, "Las columnas no están en el orden esperado"
print()
print("Todas las validaciones pasaron.")

Tamaño: (782, 9)
Celdas vacías: 0
Filas repetidas: 0
Combinaciones país-año repetidas: 0
Valores negativos: 0

Filas por año:
year
2015    158
2016    157
2017    155
2018    156
2019    156
Name: count, dtype: int64

Todas las validaciones pasaron.


## 9. Guardar el dataset unificado

In [8]:
salida = f"{PROCESSED}/happiness_unified.csv"
unificado.to_csv(salida, index=False)
print("Guardado en:", salida)

# Se vuelve a leer para comprobar que quedó bien escrito
pd.read_csv(salida).head()

Guardado en: ../data/processed/happiness_unified.csv


,country,year,gdp,family,health,freedom,generosity,corruption,happiness_score
0,Switzerland,2015,1.39651,1.34951,0.94143,0.66557,0.29678,0.41978,7.587
1,Iceland,2015,1.30232,1.40223,0.94784,0.62877,0.43630,0.14145,7.561
2,Denmark,2015,1.32548,1.36058,0.87464,0.64938,0.34139,0.48357,7.527
3,Norway,2015,1.45900,1.33095,0.88521,0.66973,0.34699,0.36503,7.522
4,Canada,2015,1.32629,1.32261,0.90563,0.63297,0.45811,0.32957,7.427


## 10. Resumen de decisiones de limpieza

| # | Problema encontrado en el EDA | Decisión | Justificación |
|---|---|---|---|
| 1 | 3 estilos de nombres de columnas | Diccionario de equivalencias por familia de esquema | Explícito, auditable y alineado con el JSON de Kafka |
| 2 | Columnas que solo existen en algunos años | Conservar solo las comunes a los 5 años | Las demás dejarían años completos vacíos |
| 3 | Ranking y residuo de distopía | Descartar | Se derivan del puntaje: fuga del objetivo |
| 4 | No hay columna de año | Crearla desde el nombre del archivo | Sin ella no se distingue el año al unir |
| 5 | 6 países escritos de dos formas | Unificar a un solo nombre | El país es clave en Kafka y dimensión en la base de datos |
| 6 | 1 celda vacía (corrupción, UAE, 2018) | Rellenar con la mediana de 2018 | Conserva el país; la mediana resiste los atípicos |
| 7 | Exceso de decimales en 2017 | Redondear a 5 decimales | Mayor precisión real de las fuentes; no se pierde información |
| 8 | Valores atípicos | Conservar | Son valores reales, no errores |
| 9 | 31 ceros exactos | Conservar | Son el mínimo del indicador, no faltantes |

**Resultado:** un dataset de **782 filas y 9 columnas**, sin celdas vacías ni duplicados, listo para la ingeniería de características (Paso 3).